In [21]:
sentences = [
    "Barack Obama was born in Hawaii",
    "Google is based in Mountain View",

    "Elon Musk founded SpaceX",
    "SpaceX was founded by Elon Musk",

    "Microsoft was founded by Bill Gates",
    "Bill Gates founded Microsoft",

    "Amazon is located in Seattle",
    "Seattle is where Amazon is located",

    "Facebook was created by Mark Zuckerberg",
    "Mark Zuckerberg created Facebook",

    "Tesla is a company",
    "Tesla is known as a company",

    "Apple is headquartered in Cupertino",
    "Cupertino is where Apple is headquartered",

    "Sundar Pichai works at Google",
    "Google employs Sundar Pichai",

    "Jeff Bezos founded Amazon",
    "Amazon was founded by Jeff Bezos",

    "Larry Page co founded Google",
    "Google was co founded by Larry Page",

    "Sergey Brin co founded Google",
    "Google was co founded by Sergey Brin",

    "Netflix is based in California",
    "California is where Netflix is based",

    "IBM is an old company",
    "IBM is known as an old company",

    "Intel is located in Santa Clara",
    "Santa Clara is where Intel is located",

    "Twitter is used worldwide",
    "Twitter is used across the world",

    "Jack Dorsey founded Twitter",
    "Twitter was founded by Jack Dorsey",

    "Nvidia is a tech company",
    "Nvidia is known as a tech company",

    "OpenAI is based in San Francisco",
    "San Francisco is where OpenAI is based",

    "Sam Altman leads OpenAI",
    "OpenAI is led by Sam Altman",

    "Oracle is headquartered in Texas",
    "Texas is where Oracle is headquartered",

    "Uber operates globally",
    "Uber is operated globally"
]


labels = [
    ["PERSON","PERSON","O","O","O","LOCATION"],
    ["ORGANIZATION","O","O","O","LOCATION","LOCATION"],

    ["PERSON","PERSON","O","ORGANIZATION"],
    ["ORGANIZATION","O","O","O","PERSON","PERSON"],

    ["ORGANIZATION","O","O","O","PERSON","PERSON"],
    ["PERSON","PERSON","O","ORGANIZATION"],

    ["ORGANIZATION","O","O","O","LOCATION"],
    ["LOCATION","O","O","ORGANIZATION","O","O"],

    ["ORGANIZATION","O","O","O","PERSON","PERSON"],
    ["PERSON","PERSON","O","ORGANIZATION"],

    ["ORGANIZATION","O","O","O"],
    ["ORGANIZATION","O","O","O","O"],

    ["ORGANIZATION","O","O","O","LOCATION"],
    ["LOCATION","O","O","ORGANIZATION","O","O"],

    ["PERSON","PERSON","O","O","ORGANIZATION"],
    ["ORGANIZATION","O","PERSON","PERSON"],

    ["PERSON","PERSON","O","ORGANIZATION"],
    ["ORGANIZATION","O","O","O","PERSON","PERSON"],

    ["PERSON","PERSON","O","O","ORGANIZATION"],
    ["ORGANIZATION","O","O","O","PERSON","PERSON"],

    ["PERSON","PERSON","O","O","ORGANIZATION"],
    ["ORGANIZATION","O","O","O","PERSON","PERSON"],

    ["ORGANIZATION","O","O","O","LOCATION"],
    ["LOCATION","O","O","ORGANIZATION","O","O"],

    ["ORGANIZATION","O","O","O","O"],
    ["ORGANIZATION","O","O","O","O","O"],

    ["ORGANIZATION","O","O","O","LOCATION","LOCATION"],
    ["LOCATION","LOCATION","O","O","ORGANIZATION","O","O"],

    ["ORGANIZATION","O","O","O"],
    ["ORGANIZATION","O","O","O","O","O"],

    ["PERSON","PERSON","O","ORGANIZATION"],
    ["ORGANIZATION","O","O","O","PERSON","PERSON"],

    ["ORGANIZATION","O","O","O","O"],
    ["ORGANIZATION","O","O","O","O","O"],

    ["ORGANIZATION","O","O","O","LOCATION","LOCATION"],
    ["LOCATION","LOCATION","O","O","ORGANIZATION","O","O"],

    ["PERSON","PERSON","O","ORGANIZATION"],
    ["ORGANIZATION","O","O","PERSON","PERSON"],

    ["ORGANIZATION","O","O","O","LOCATION"],
    ["LOCATION","O","O","ORGANIZATION","O","O"],

    ["ORGANIZATION","O","O"],
    ["ORGANIZATION","O","O"]
]

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------
# Build Vocabulary
# ----------------------------
# word2idx = {"<PAD>":0}
word2idx = {"<PAD>":0, "<UNK>":1}
label2idx = {"O":0, "PERSON":1, "LOCATION":2, "ORGANIZATION":3}

for sent in sentences:
    for word in sent.split():
        if word not in word2idx:
            word2idx[word] = len(word2idx)

idx2label = {v:k for k,v in label2idx.items()}

# ----------------------------
# Encode Data
# ----------------------------
max_len = max(len(s.split()) for s in sentences)

# def encode_sentence(sentence):
#     tokens = sentence.split()
#     ids = [word2idx[w] for w in tokens]
#     ids += [0] * (max_len - len(ids))
#     return ids

def encode_sentence(sentence):
    tokens = sentence.split()
    ids = [word2idx.get(w, word2idx["<UNK>"]) for w in tokens]
    ids += [0] * (max_len - len(ids))
    return ids

def encode_labels(lbls):
    ids = [label2idx[l] for l in lbls]
    ids += [0] * (max_len - len(ids))
    return ids

X = torch.tensor([encode_sentence(s) for s in sentences])
y = torch.tensor([encode_labels(l) for l in labels])


In [23]:

# ----------------------------
# Model
# ----------------------------
class RNN_NER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

model = RNN_NER(len(word2idx), 32, 64, len(label2idx))


In [24]:

# ----------------------------
# Training
# ----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 50

for epoch in range(epochs):
    outputs = model(X)

    loss = criterion(outputs.view(-1, len(label2idx)), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.2198
Epoch 20, Loss: 0.0218
Epoch 30, Loss: 0.0030
Epoch 40, Loss: 0.0010
Epoch 50, Loss: 0.0006


In [25]:
def predict(sentence):
    model.eval()
    with torch.no_grad():
        encoded = torch.tensor([encode_sentence(sentence)])
        outputs = model(encoded)
        preds = torch.argmax(outputs, dim=-1).squeeze().tolist()

        words = sentence.split()

        for w, p in zip(words, preds):
            print(f"{w} -> {idx2label[p]}")

# Test
predict("Elon Musk works at Tesla")

Elon -> PERSON
Musk -> PERSON
works -> O
at -> O
Tesla -> ORGANIZATION


**Observation**


earlier, if a token was added that was not in the dataset the model would crash. So an <UNK> token was added. Now it just guesses what label comes next when it sees an unseen word like in the case for "lives"

In [27]:
predict("Elon Musk lives in Texas")

Elon -> PERSON
Musk -> PERSON
lives -> PERSON
in -> O
Texas -> LOCATION


Decreasing no. of epochs

In [28]:

# ----------------------------
# Training
# ----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 10

for epoch in range(epochs):
    outputs = model(X)

    loss = criterion(outputs.view(-1, len(label2idx)), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.0000


In [30]:
predict("Elon Musk works at Oracle")

Elon -> PERSON
Musk -> PERSON
works -> O
at -> O
Oracle -> ORGANIZATION


In [ ]:
# lets give it something nonsensical

In [31]:
predict("Bill Musk met at Oracle Texas")

Bill -> PERSON
Musk -> PERSON
met -> PERSON
at -> O
Oracle -> ORGANIZATION
Texas -> LOCATION


# Changing number of units
increased it to 128

In [32]:

# ----------------------------
# Model
# ----------------------------
class RNN_NER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

model = RNN_NER(len(word2idx), 32, 128, len(label2idx))


In [36]:

# ----------------------------
# Training
# ----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 50

for epoch in range(epochs):
    outputs = model(X)

    loss = criterion(outputs.view(-1, len(label2idx)), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.0325
Epoch 20, Loss: 0.0029
Epoch 30, Loss: 0.0007
Epoch 40, Loss: 0.0003
Epoch 50, Loss: 0.0002


In [37]:
predict("Elon Musk works at Oracle")

Elon -> PERSON
Musk -> PERSON
works -> O
at -> ORGANIZATION
Oracle -> ORGANIZATION


In [39]:
predict("Bill Musk met at Oracle Texas")

Bill -> PERSON
Musk -> PERSON
met -> LOCATION
at -> ORGANIZATION
Oracle -> ORGANIZATION
Texas -> O


you can see in the cell above that it terms 'at '  as an organization and 'texas' as O. This means model performance has gone down

In [40]:
predict("Bill went to Microsoft and Elon works at Tesla")

Bill -> PERSON
went -> PERSON
to -> O
Microsoft -> ORGANIZATION
and -> O
Elon -> PERSON
works -> O
at -> O
Tesla -> ORGANIZATION


Here it works better. Went was not in the dataset so it being termed as person was the model guessing.